# TinyPress — Prompt Compression Engine

**HuggingFace Build Small Hackathon · Track: Thousand Token Wood**

| Layer | Detail |
|-------|--------|
| Compression | `Qwen/Qwen2.5-1.5B-Instruct` (default, switchable) |
| Scoring | `sentence-transformers/all-MiniLM-L6-v2` (default, switchable) |
| UI | Gradio 5 — public share URL |
| Storage | SQLite at `/content/tinypress.db` |

**Features**
- Compress text to a user-defined token budget
- Live 🔴 / 🟢 compression readiness banner
- Per-token colour highlight panel (toggle on/off)
- Dynamic compression model switching (5 curated <32B models)
- Dynamic scoring embedder switching (6 models, with per-model impact info)
- 👍 / 👎 feedback on every compression result, with optional text comment
- Compression run history persisted to SQLite
- Column picker in History tab — compact default view, expandable to all fields
- Per-row delete in history
- Side-by-side word-level diff viewer with feedback badge and token detail

> **Recommended runtime:** GPU  →  Runtime → Change runtime type → T4 GPU

---

### About the author

Built by **Sriharsha C R** — AI Engineer, Cloud Native developer, and knowledge sharer.
If this was useful, feel free to connect — always happy to chat about AI, LLMs, or anything in between.

[![LinkedIn](https://img.shields.io/badge/LinkedIn-sriharsha--cr-0a66c2?logo=linkedin&logoColor=white)](https://www.linkedin.com/in/sriharsha-cr)
[![X / Twitter](https://img.shields.io/badge/X-@sriharsha__cr-000000?logo=x&logoColor=white)](https://x.com/sriharsha_cr)
[![HuggingFace](https://img.shields.io/badge/HuggingFace-sriharsha--cr-ff9d00?logo=huggingface&logoColor=white)](https://huggingface.co/sriharsha-cr)
[![GitHub](https://img.shields.io/badge/GitHub-SriharshaCR-181717?logo=github&logoColor=white)](https://github.com/SriharshaCR)

## Step 1 — Install dependencies

In [ ]:
!pip install -q \
    "gradio==5.0" \
    "transformers>=4.40.0" \
    "sentence-transformers>=3.0.0" \
    "torch>=2.2.0" \
    "numpy>=1.26.0" \
    "pandas>=2.0.0" \
    "accelerate>=0.30.0" \
    "huggingface_hub==0.25.2"

## Step 2 — Runtime check

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if device == 'cuda' else torch.float32

print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'dtype  : {dtype}')

## Step 3 — Configuration

In [3]:
# Curated <32B open-weight causal LMs for local / Colab inference.
AVAILABLE_MODELS = [
    'Qwen/Qwen2.5-1.5B-Instruct',
    'Qwen/Qwen2.5-0.5B-Instruct',
    'HuggingFaceTB/SmolLM2-1.7B-Instruct',
    'microsoft/Phi-3.5-mini-instruct',
    'meta-llama/Llama-3.2-1B-Instruct',
]

# Curated sentence-transformer embedding models for quality scoring.
AVAILABLE_EMBEDDER_MODELS = [
    'sentence-transformers/all-MiniLM-L6-v2',
    'sentence-transformers/all-mpnet-base-v2',
    'BAAI/bge-small-en-v1.5',
    'BAAI/bge-base-en-v1.5',
    'mixedbread-ai/mxbai-embed-large-v1',
    'Alibaba-NLP/gte-Qwen2-1.5B-instruct',
]

EMBEDDER_INFO = {
    'sentence-transformers/all-MiniLM-L6-v2': (
        '⚡ **Fast · 22M params · Default**  \n'
        'Great baseline. Scores are reliable for typical compression ratios. '
        'Runs comfortably on CPU — minimal overhead.'
    ),
    'sentence-transformers/all-mpnet-base-v2': (
        '⚖️ **Balanced · 110M params**  \n'
        'Noticeably sharper quality scores than MiniLM, especially on longer texts. '
        'Small speed trade-off; fine on CPU.'
    ),
    'BAAI/bge-small-en-v1.5': (
        '⚡ **Fast · 33M params**  \n'
        'Strong quality-to-size ratio — often matches MiniLM on accuracy while being '
        'slightly more sensitive to meaning shifts. Good CPU option.'
    ),
    'BAAI/bge-base-en-v1.5': (
        '⚖️ **Balanced · 109M params**  \n'
        'Consistently strong on semantic similarity benchmarks. '
        'Scores will be more discriminating — small differences in compression quality show up more clearly.'
    ),
    'mixedbread-ai/mxbai-embed-large-v1': (
        '🏆 **High quality · 335M params**  \n'
        'Top-tier similarity scores. Quality readings will be the most accurate here, '
        'but slower to load and run. GPU recommended.'
    ),
    'Alibaba-NLP/gte-Qwen2-1.5B-instruct': (
        '🔬 **Best quality · 1.5B params**  \n'
        'Strongest semantic understanding in this list. Scores will reflect subtle meaning loss '
        'that smaller models miss. Requires significant RAM/VRAM — GPU strongly recommended.'
    ),
}

In [ ]:
import os

LLM_MODEL      = os.getenv('LLM_MODEL',      AVAILABLE_MODELS[1])
EMBEDDER_MODEL = os.getenv('EMBEDDER_MODEL', AVAILABLE_EMBEDDER_MODELS[0])
DB_PATH        = os.getenv('DB_PATH',        '/content/tinypress.db')
SERVER_PORT    = int(os.getenv('PORT', 7860))

DEFAULT_TARGET_TOKENS = 500
MAX_NEW_TOKENS        = 1024
APP_TITLE             = 'TinyPress'

PUBLIC_UI = True

print(f'LLM      : {LLM_MODEL}')
print(f'Embedder : {EMBEDDER_MODEL}')
print(f'DB       : {DB_PATH}')

## Step 4 — Model loader

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import gc

_llm               = None
_tokenizer         = None
_embedder          = None
_current_model_id    = None
_current_embedder_id = None


def get_current_model_id():
    return _current_model_id


def get_current_tokenizer_id():
    # Tokenizer is always loaded from the same HF repo as the model.
    return _current_model_id


def get_current_embedder_id():
    return _current_embedder_id


def get_llm():
    global _llm, _tokenizer
    if _llm is None:
        _load_llm(LLM_MODEL)
    return _llm, _tokenizer


def switch_llm(model_id: str) -> str:
    global _current_model_id
    if _current_model_id == model_id:
        return f'Already using {model_id}'
    _unload_llm()
    _load_llm(model_id)
    return f'Loaded: {model_id}'


def _load_llm(model_id: str):
    """Load model + its paired tokenizer. Both come from the same model_id."""
    global _llm, _tokenizer, _current_model_id
    print(f'Loading LLM: {model_id} ...')
    _tokenizer = AutoTokenizer.from_pretrained(model_id)
    _llm = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=dtype,
        device_map='auto',
    )
    _llm.eval()
    _current_model_id = model_id
    print(f'LLM ready: {model_id}')


def _unload_llm():
    """Free GPU/CPU memory before loading a different model."""
    global _llm, _tokenizer, _current_model_id
    del _llm, _tokenizer
    _llm = None
    _tokenizer = None
    _current_model_id = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def get_embedder():
    global _embedder, _current_embedder_id
    if _embedder is None:
        _load_embedder(EMBEDDER_MODEL)
    return _embedder


def switch_embedder(model_id: str) -> str:
    global _current_embedder_id
    if _current_embedder_id == model_id:
        return f'Already using {model_id}'
    _unload_embedder()
    _load_embedder(model_id)
    return f'Loaded: {model_id}'


def _load_embedder(model_id: str):
    global _embedder, _current_embedder_id
    print(f'Loading embedder: {model_id} ...')
    # Explicitly set device to 'cpu' to avoid ZeroGPU conflicts
    _embedder = SentenceTransformer(model_id, device='cpu')
    _current_embedder_id = model_id
    print(f'Embedder ready: {model_id}')


def _unload_embedder():
    global _embedder, _current_embedder_id
    del _embedder
    _embedder = None
    _current_embedder_id = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print('Model loader defined.')

## Step 5 — Core pipeline

In [ ]:
import numpy as np

# ── tokenizer utils ───────────────────────────────────────────────────────────

def count_tokens(text: str) -> int:
    _, tokenizer = get_llm()
    return len(tokenizer.encode(text, add_special_tokens=False))


def get_token_strings(text: str) -> list:
    """Return the decoded surface string for every token in text."""
    _, tokenizer = get_llm()
    ids = tokenizer.encode(text, add_special_tokens=False)
    return [tokenizer.decode([i]) for i in ids]


# ── compressor ────────────────────────────────────────────────────────────────

_PROMPT_TEMPLATE = (
    'You are a lossless compression assistant. '
    'Compress the following text to at most {target} tokens.\n'
    'Preserve all key facts, decisions, and intent. '
    'Do not add commentary. Output only the compressed text.\n\n'
    'TEXT:\n{text}\n\nCOMPRESSED:'
)


def _generate(prompt: str) -> str:
    model, tokenizer = get_llm()
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def compress(text: str, target_tokens: int) -> tuple:
    """Returns (compressed_text, input_token_count, output_token_count)."""
    input_tokens = count_tokens(text)
    if input_tokens <= target_tokens:
        return text, input_tokens, input_tokens

    prompt     = _PROMPT_TEMPLATE.format(target=target_tokens, text=text)
    compressed = _generate(prompt)

    # Hard-trim if model overshoots.
    _, tokenizer = get_llm()
    ids = tokenizer.encode(compressed, add_special_tokens=False)
    if len(ids) > target_tokens:
        compressed = tokenizer.decode(ids[:target_tokens], skip_special_tokens=True)

    output_tokens = count_tokens(compressed)
    return compressed, input_tokens, output_tokens


# ── scorer ────────────────────────────────────────────────────────────────────

def semantic_score(original: str, compressed: str) -> float:
    embedder = get_embedder()
    vecs = embedder.encode([original, compressed], convert_to_numpy=True)
    cos  = float(
        np.dot(vecs[0], vecs[1]) / (np.linalg.norm(vecs[0]) * np.linalg.norm(vecs[1]))
    )
    return round(max(0.0, min(1.0, cos)), 4)


print('Core pipeline defined.')

## Step 6 — Diff renderer

In [ ]:
import difflib
import html as _h


def _word_diff(original: str, compressed: str) -> tuple:
    """
    Word-level SequenceMatcher diff.
    Returns (annotated_original_html, annotated_compressed_html).
    Colour key:
      original  — red strikethrough = dropped
      compressed — amber            = rewritten
      compressed — green            = inserted
      plain                         = unchanged
    """
    orig_words = original.split()
    comp_words = compressed.split()
    matcher = difflib.SequenceMatcher(None, orig_words, comp_words, autojunk=False)

    orig_parts, comp_parts = [], []

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        ow = _h.escape(' '.join(orig_words[i1:i2]))
        cw = _h.escape(' '.join(comp_words[j1:j2]))

        if tag == 'equal':
            orig_parts.append(ow)
            comp_parts.append(cw)

        elif tag == 'delete':
            orig_parts.append(
                f'<mark style="background:#fee2e2;color:#b91c1c;'
                f'text-decoration:line-through;padding:1px 3px;border-radius:3px">{ow}</mark>'
            )

        elif tag == 'insert':
            comp_parts.append(
                f'<mark style="background:#dcfce7;color:#15803d;'
                f'padding:1px 3px;border-radius:3px">{cw}</mark>'
            )

        elif tag == 'replace':
            orig_parts.append(
                f'<mark style="background:#fee2e2;color:#b91c1c;'
                f'text-decoration:line-through;padding:1px 3px;border-radius:3px">{ow}</mark>'
            )
            comp_parts.append(
                f'<mark style="background:#fef9c3;color:#92400e;'
                f'padding:1px 3px;border-radius:3px">{cw}</mark>'
            )

    return ' '.join(orig_parts), ' '.join(comp_parts)


def render_diff_html(record: dict) -> str:
    """Build a self-contained side-by-side diff HTML block for a compression run."""
    original   = record.get('input_text', '')
    compressed = record.get('output_text', '')
    if not original or not compressed:
        return ''

    orig_html, comp_html = _word_diff(original, compressed)

    model      = _h.escape(record.get('model', '—'))
    tokenizer  = _h.escape(record.get('tokenizer', '—'))
    ts         = _h.escape(record.get('timestamp', '—'))
    in_tok     = record.get('input_tokens',  '—')
    out_tok    = record.get('output_tokens', '—')
    target_tok = record.get('target_tokens', '—')
    ratio      = record.get('compression_ratio', 0)
    quality    = record.get('quality_score', 0)
    duration   = record.get('duration_ms', '—')
    run_id     = record.get('id', '—')

    feedback_val  = record.get('feedback')
    feedback_note = _h.escape(record.get('feedback_comment') or '')

    # Build optional feedback block
    if feedback_val is not None:
        badge_bg    = '#f0fdf4' if feedback_val == 1 else '#fef2f2'
        badge_color = '#15803d' if feedback_val == 1 else '#b91c1c'
        badge_text  = '👍 Helpful' if feedback_val == 1 else '👎 Not helpful'
        feedback_block = (
            f'<div style="display:flex;flex-wrap:wrap;align-items:center;gap:8px;'
            f'margin-top:10px;padding:8px 12px;border-radius:6px;background:{badge_bg}">'
            f'<span style="font-weight:600;font-size:0.8rem;color:{badge_color}">{badge_text}</span>'
        )
        if feedback_note:
            feedback_block += (
                f'<span style="font-size:0.8rem;color:#374151;font-style:italic">'
                f'"{feedback_note}"</span>'
            )
        feedback_block += '</div>'
    else:
        feedback_block = ''

    return f"""
<div style="font-family:system-ui,sans-serif;margin-top:4px">

  <!-- Primary meta chips -->
  <div style="display:flex;flex-wrap:wrap;gap:6px;margin-bottom:6px;font-size:0.78rem">
    <span style="background:#f3f4f6;padding:3px 9px;border-radius:12px;color:#374151">Run #{run_id}</span>
    <span style="background:#f3f4f6;padding:3px 9px;border-radius:12px;color:#374151">{ts}</span>
    <span style="background:#eff6ff;padding:3px 9px;border-radius:12px;color:#1d4ed8">{model}</span>
    <span style="background:#f0fdf4;padding:3px 9px;border-radius:12px;color:#15803d">Quality {quality:.4f}</span>
    <span style="background:#fff7ed;padding:3px 9px;border-radius:12px;color:#c2410c">Ratio {ratio:.4f}</span>
    <span style="background:#faf5ff;padding:3px 9px;border-radius:12px;color:#7e22ce">&#9201; {duration} ms</span>
  </div>

  <!-- Secondary meta chips -->
  <div style="display:flex;flex-wrap:wrap;gap:6px;margin-bottom:12px;font-size:0.78rem">
    <span style="background:#f3f4f6;padding:3px 9px;border-radius:12px;color:#374151">{in_tok} in → {out_tok} out (target {target_tok})</span>
    <span style="background:#f3f4f6;padding:3px 9px;border-radius:12px;color:#374151">tokenizer: {tokenizer}</span>
  </div>

  <!-- Side-by-side panels -->
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px">
    <div style="border:1px solid #fecaca;border-radius:8px;overflow:hidden">
      <div style="background:#fef2f2;padding:8px 14px;border-bottom:1px solid #fecaca;
                  display:flex;justify-content:space-between;align-items:center">
        <span style="font-weight:700;font-size:0.8rem;color:#b91c1c">ORIGINAL</span>
        <span style="font-size:0.75rem;color:#6b7280">{in_tok} tokens</span>
      </div>
      <div style="padding:14px;line-height:1.8;font-size:0.875rem;color:#1a1a1a;
                  max-height:340px;overflow-y:auto;word-break:break-word">{orig_html}</div>
    </div>
    <div style="border:1px solid #bbf7d0;border-radius:8px;overflow:hidden">
      <div style="background:#f0fdf4;padding:8px 14px;border-bottom:1px solid #bbf7d0;
                  display:flex;justify-content:space-between;align-items:center">
        <span style="font-weight:700;font-size:0.8rem;color:#15803d">COMPRESSED</span>
        <span style="font-size:0.75rem;color:#6b7280">{out_tok} tokens</span>
      </div>
      <div style="padding:14px;line-height:1.8;font-size:0.875rem;color:#1a1a1a;
                  max-height:340px;overflow-y:auto;word-break:break-word">{comp_html}</div>
    </div>
  </div>

  {feedback_block}

  <!-- Legend -->
  <div style="display:flex;flex-wrap:wrap;gap:14px;margin-top:10px;font-size:0.75rem;color:#6b7280;align-items:center">
    <mark style="background:#fee2e2;color:#b91c1c;text-decoration:line-through;padding:2px 7px;border-radius:3px">dropped</mark>
    <mark style="background:#fef9c3;color:#92400e;padding:2px 7px;border-radius:3px">rewritten</mark>
    <mark style="background:#dcfce7;color:#15803d;padding:2px 7px;border-radius:3px">inserted</mark>
    <span>plain = unchanged</span>
  </div>

</div>
"""


print('Diff renderer defined.')

## Step 7 — Database

In [ ]:
import sqlite3

_SCHEMA = """
CREATE TABLE IF NOT EXISTS compression_runs (
    id                INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp         TEXT    NOT NULL,
    model             TEXT    NOT NULL,
    tokenizer         TEXT    NOT NULL,
    input_tokens      INTEGER NOT NULL,
    output_tokens     INTEGER NOT NULL,
    target_tokens     INTEGER NOT NULL,
    compression_ratio REAL    NOT NULL,
    quality_score     REAL    NOT NULL,
    duration_ms       REAL    NOT NULL,
    input_text        TEXT    NOT NULL,
    output_text       TEXT    NOT NULL,
    feedback          INTEGER,
    feedback_comment  TEXT
);
"""


def _connect():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def init_db():
    conn = _connect()
    conn.executescript(_SCHEMA)
    for col, typedef in [
        ('tokenizer',        'TEXT NOT NULL DEFAULT ""'),
        ('duration_ms',      'REAL NOT NULL DEFAULT 0'),
        ('feedback',         'INTEGER'),
        ('feedback_comment', 'TEXT'),
    ]:
        try:
            conn.execute(f'ALTER TABLE compression_runs ADD COLUMN {col} {typedef}')
        except sqlite3.OperationalError:
            pass
    conn.commit()
    conn.close()


def save_run(record: dict) -> int:
    conn = _connect()
    cursor = conn.execute(
        '''
        INSERT INTO compression_runs
            (timestamp, model, tokenizer, input_tokens, output_tokens, target_tokens,
             compression_ratio, quality_score, duration_ms, input_text, output_text)
        VALUES
            (:timestamp, :model, :tokenizer, :input_tokens, :output_tokens, :target_tokens,
             :compression_ratio, :quality_score, :duration_ms, :input_text, :output_text)
        ''',
        record,
    )
    run_id = cursor.lastrowid
    conn.commit()
    conn.close()
    return run_id


def update_feedback(run_id: int, value: int):
    conn = _connect()
    conn.execute('UPDATE compression_runs SET feedback = ? WHERE id = ?', (value, run_id))
    conn.commit()
    conn.close()


def update_feedback_comment(run_id: int, comment: str):
    conn = _connect()
    conn.execute('UPDATE compression_runs SET feedback_comment = ? WHERE id = ?', (comment, run_id))
    conn.commit()
    conn.close()


def delete_run(run_id: int):
    conn = _connect()
    conn.execute('DELETE FROM compression_runs WHERE id = ?', (run_id,))
    conn.commit()
    conn.close()


def get_run(run_id: int):
    conn = _connect()
    row = conn.execute('SELECT * FROM compression_runs WHERE id = ?', (run_id,)).fetchone()
    conn.close()
    return dict(row) if row else None


def get_runs(limit: int = 100) -> list:
    conn = _connect()
    rows = conn.execute(
        'SELECT * FROM compression_runs ORDER BY id DESC LIMIT ?', (limit,)
    ).fetchall()
    conn.close()
    return [dict(r) for r in rows]


init_db()
print(f'Database ready at {DB_PATH}')

## Step 8 — Load models

Downloads and caches weights. GPU warm-cache: ~30 s. First run: a few minutes.

In [ ]:
get_llm()
get_embedder()
print('\nAll models loaded and ready.')

## Step 9 — Launch Gradio UI

Prints a **public share URL** when ready. All features are live in the UI.

In [11]:
import html as _h
import time
from datetime import datetime, timezone

import gradio as gr
import pandas as pd


# ══════════════════════════════════════════════════════════════════════════════
# COMPRESS TAB — handlers
# ══════════════════════════════════════════════════════════════════════════════

_PALETTE = [
    '#fde68a', '#bbf7d0', '#bfdbfe', '#fecaca', '#e9d5ff',
    '#fed7aa', '#99f6e4', '#e0e7ff', '#fce7f3', '#d1fae5',
]
_BTN_SHOW = '🔍  Show Token Highlights'
_BTN_HIDE = '🙈  Hide Token Highlights'


def _render_token_html(text: str) -> str:
    if not text.strip():
        return ''
    tokens = get_token_strings(text)
    if not tokens:
        return ''
    spans = []
    for i, tok in enumerate(tokens):
        color   = _PALETTE[i % len(_PALETTE)]
        display = _h.escape(tok).replace(' ', '<span style="opacity:0.35;font-size:0.7em">·</span>')
        spans.append(
            f'<span title="token {i+1}" '
            f'style="background:{color};border-radius:4px;padding:2px 5px;'
            f'font-family:\'Courier New\',monospace;font-size:0.8rem;'
            f'line-height:2.2;margin:2px 1px;display:inline-block;'
            f'cursor:default;border:1px solid rgba(0,0,0,0.06)">{display}</span>'
        )
    return (
        '<div style="font-family:system-ui,sans-serif;padding:10px 12px;'
        'border:1px solid #e5e7eb;border-radius:8px;background:#fafafa">'
        f'<div style="font-size:0.75rem;color:#6b7280;margin-bottom:8px;font-weight:500">'
        f'{len(tokens)} tokens — each chip = one token, hover for index</div>'
        '<div style="line-height:2.6;word-break:break-all;max-height:200px;overflow-y:auto">'
        + ''.join(spans) + '</div></div>'
    )


def toggle_token_panel(is_visible: bool, text: str):
    new_visible  = not is_visible
    html_content = _render_token_html(text) if new_visible else ''
    btn_label    = _BTN_HIDE if new_visible else _BTN_SHOW
    return new_visible, html_content, gr.update(value=btn_label)


def update_token_panel(text: str, is_visible: bool) -> str:
    return _render_token_html(text) if is_visible else ''


_STATUS_EMPTY = '<span></span>'
_STATUS_RED = (
    '<div style="background:#fee2e2;border:1px solid #ef4444;color:#b91c1c;'
    'padding:8px 12px;border-radius:6px;font-size:0.9rem;">'
    '🔴 <strong>Compression not needed</strong> — input ({input_tok} tokens) '
    'is already within the {budget}-token budget.</div>'
)
_STATUS_GREEN = (
    '<div style="background:#dcfce7;border:1px solid #22c55e;color:#15803d;'
    'padding:8px 12px;border-radius:6px;font-size:0.9rem;">'
    '🟢 <strong>Ready to compress</strong> — {input_tok} tokens → {budget} token budget '
    '({delta} tokens to shed).</div>'
)


def compression_status(text: str, target_tokens: int) -> str:
    if not text.strip():
        return _STATUS_EMPTY
    n = count_tokens(text)
    if n <= int(target_tokens):
        return _STATUS_RED.format(input_tok=n, budget=int(target_tokens))
    return _STATUS_GREEN.format(input_tok=n, budget=int(target_tokens), delta=n - int(target_tokens))


def run_compression(text: str, target_tokens: int):
    _hidden = gr.update(visible=False)
    if not text.strip():
        return ('', 0, 0, 0, 0.0, None,
                _hidden, _hidden, gr.update(value='', visible=False),
                gr.update(value='', visible=False), _hidden, gr.update(value='', visible=False))

    t0 = time.perf_counter()
    compressed, input_tokens, output_tokens = compress(text, int(target_tokens))
    duration_ms = round((time.perf_counter() - t0) * 1000, 1)

    ratio   = round(output_tokens / input_tokens, 4) if input_tokens else 0.0
    quality = semantic_score(text, compressed)

    run_id = save_run({
        'timestamp':         datetime.now(timezone.utc).isoformat(),
        'model':             get_current_model_id() or LLM_MODEL,
        'tokenizer':         get_current_tokenizer_id() or LLM_MODEL,
        'input_tokens':      input_tokens,
        'output_tokens':     output_tokens,
        'target_tokens':     int(target_tokens),
        'compression_ratio': ratio,
        'quality_score':     quality,
        'duration_ms':       duration_ms,
        'input_text':        text,
        'output_text':       compressed,
    })

    return (
        compressed, input_tokens, output_tokens, ratio, quality,
        run_id,
        gr.update(visible=True), gr.update(visible=True),
        gr.update(value='', visible=True),
        gr.update(value='', visible=False),
        gr.update(visible=False),
        gr.update(value='', visible=False),
    )


def load_model(model_id: str) -> str:
    if not model_id:
        return 'No model selected.'
    try:
        return switch_llm(model_id)
    except Exception as exc:
        return f'Error loading {model_id}: {exc}'


def load_embedder(model_id: str) -> str:
    if not model_id:
        return 'No model selected.'
    try:
        return switch_embedder(model_id)
    except Exception as exc:
        return f'Error loading {model_id}: {exc}'


def on_embedder_change(model_id: str) -> str:
    return EMBEDDER_INFO.get(model_id, '')


def submit_feedback(run_id, value: int):
    if run_id is None:
        return 'Run a compression first.', gr.update(visible=False), gr.update(visible=False), gr.update(value='', visible=False)
    update_feedback(run_id, value)
    msg = '👍 Marked as helpful — thanks!' if value == 1 else '👎 Noted — thanks for the feedback!'
    return msg, gr.update(visible=True), gr.update(visible=True), gr.update(value='', visible=False)


def save_comment(run_id, comment: str):
    if run_id is None:
        return gr.update(value='Run a compression first.', visible=True)
    if not comment.strip():
        return gr.update(value='Type a note first.', visible=True)
    update_feedback_comment(run_id, comment.strip())
    return gr.update(value='✓ Note saved.', visible=True)


# ══════════════════════════════════════════════════════════════════════════════
# HISTORY TAB — handlers
# ══════════════════════════════════════════════════════════════════════════════

_DEFAULT_COLS = ['id', 'timestamp', 'model', 'compression_ratio', 'quality_score', 'feedback']
_ALL_COLS = [
    'id', 'timestamp', 'model', 'tokenizer',
    'input_tokens', 'output_tokens', 'target_tokens',
    'compression_ratio', 'quality_score', 'duration_ms',
    'feedback', 'feedback_comment',
]


def load_history(selected_cols=None):
    cols = selected_cols if selected_cols else _DEFAULT_COLS
    runs = get_runs(limit=100)
    if not runs:
        return pd.DataFrame(columns=cols), '', '', ''
    df       = pd.DataFrame(runs)
    existing = [c for c in cols if c in df.columns]
    df       = df[existing]
    avg_q = f"{df['quality_score'].mean():.4f}" if 'quality_score' in df.columns else '—'
    avg_r = f"{df['compression_ratio'].mean():.4f}" if 'compression_ratio' in df.columns else '—'
    return df, avg_q, avg_r, ''


def on_row_select(evt: gr.SelectData, df: pd.DataFrame):
    if df is None or df.empty:
        return None, '', 'No rows available.'
    row_idx = evt.index[0]
    run_id  = int(df.iloc[row_idx]['id'])
    record  = get_run(run_id)
    if not record:
        return None, '', f'Row {run_id} not found in database.'
    return run_id, render_diff_html(record), f'Row {run_id} selected — click Delete to remove.'


def delete_selected(run_id, selected_cols):
    if run_id is None:
        df, avg_q, avg_r, _ = load_history(selected_cols)
        return df, avg_q, avg_r, None, '', 'No row selected.'
    delete_run(run_id)
    df, avg_q, avg_r, _ = load_history(selected_cols)
    return df, avg_q, avg_r, None, '', f'Row {run_id} deleted.'


# ══════════════════════════════════════════════════════════════════════════════
# BUILD APP
# ══════════════════════════════════════════════════════════════════════════════

def build_app() -> gr.Blocks:
    with gr.Blocks(title=APP_TITLE) as app:

        # ── Compress tab ──────────────────────────────────────────────────
        with gr.Tab('Compress'):
            gr.Markdown('## TinyPress — Prompt Compression Engine')
            gr.Markdown(
                'Paste any long text. Set your token budget. Get a compressed version '
                'that preserves intent — scored for quality.'
            )

            with gr.Accordion('Model Settings', open=False):
                gr.Markdown('**Compression Model**')
                model_dropdown = gr.Dropdown(
                    choices=AVAILABLE_MODELS, value=LLM_MODEL,
                    label='Compression Model', allow_custom_value=True,
                )
                load_model_btn = gr.Button('Load Model', variant='secondary')
                model_status   = gr.Textbox(label='Model Status', value=f'Active: {LLM_MODEL}', interactive=False)

                gr.Markdown("---") # Replaced gr.Divider() with gr.Markdown("---")

                gr.Markdown('**Scoring Embedder**')
                embedder_dropdown = gr.Dropdown(
                    choices=AVAILABLE_EMBEDDER_MODELS, value=EMBEDDER_MODEL,
                    label='Embedder Model', allow_custom_value=True,
                )
                embedder_info_panel = gr.Markdown(value=EMBEDDER_INFO.get(EMBEDDER_MODEL, ''))
                load_embedder_btn = gr.Button('Load Embedder', variant='secondary')
                embedder_status   = gr.Textbox(label='Embedder Status', value=f'Active: {EMBEDDER_MODEL}', interactive=False)

            with gr.Row():
                with gr.Column():
                    input_text = gr.Textbox(label='Input Text', lines=12, placeholder='Paste your text here...')
                    token_toggle_btn = gr.Button(_BTN_SHOW, variant='secondary', size='sm')
                    token_panel      = gr.HTML(value='')
                    tokens_visible   = gr.State(value=False)
                    target_slider    = gr.Slider(minimum=100, maximum=1000, value=DEFAULT_TARGET_TOKENS, step=50, label='Target Token Budget')
                    status_banner    = gr.HTML(value=_STATUS_EMPTY)
                    compress_btn     = gr.Button('Compress', variant='primary')

                with gr.Column():
                    output_text = gr.Textbox(label='Compressed Output', lines=12)
                    with gr.Row():
                        input_tok  = gr.Number(label='Input Tokens',  interactive=False)
                        output_tok = gr.Number(label='Output Tokens', interactive=False)
                    with gr.Row():
                        ratio   = gr.Number(label='Compression Ratio',   interactive=False)
                        quality = gr.Number(label='Quality Score (0–1)', interactive=False)
                    gr.Markdown('**Was this compression helpful?**')
                    with gr.Row():
                        thumbs_up_btn   = gr.Button('👍  Helpful',     variant='secondary', visible=False, scale=1)
                        thumbs_down_btn = gr.Button('👎  Not helpful', variant='secondary', visible=False, scale=1)
                    feedback_status  = gr.Markdown('', visible=False)
                    comment_box      = gr.Textbox(
                        label='Add a note (optional)',
                        placeholder="e.g. 'lost key dates', 'too short', 'great summary'",
                        lines=2, visible=False,
                    )
                    save_comment_btn = gr.Button('Save note', variant='secondary', size='sm', visible=False)
                    comment_saved    = gr.Markdown('', visible=False)

            last_run_id = gr.State(value=None)

            token_toggle_btn.click(fn=toggle_token_panel, inputs=[tokens_visible, input_text], outputs=[tokens_visible, token_panel, token_toggle_btn])
            input_text.change(fn=update_token_panel, inputs=[input_text, tokens_visible], outputs=[token_panel])
            _sa = dict(inputs=[input_text, target_slider], outputs=[status_banner])
            input_text.change(fn=compression_status, **_sa)
            target_slider.change(fn=compression_status, **_sa)
            load_model_btn.click(fn=load_model, inputs=[model_dropdown], outputs=[model_status])
            embedder_dropdown.change(fn=on_embedder_change, inputs=[embedder_dropdown], outputs=[embedder_info_panel])
            load_embedder_btn.click(fn=load_embedder, inputs=[embedder_dropdown], outputs=[embedder_status])
            compress_btn.click(
                fn=run_compression,
                inputs=[input_text, target_slider],
                outputs=[output_text, input_tok, output_tok, ratio, quality,
                         last_run_id, thumbs_up_btn, thumbs_down_btn, feedback_status,
                         comment_box, save_comment_btn, comment_saved],
            )
            thumbs_up_btn.click(
                fn=lambda run_id: submit_feedback(run_id, 1),
                inputs=[last_run_id],
                outputs=[feedback_status, comment_box, save_comment_btn, comment_saved],
            )
            thumbs_down_btn.click(
                fn=lambda run_id: submit_feedback(run_id, -1),
                inputs=[last_run_id],
                outputs=[feedback_status, comment_box, save_comment_btn, comment_saved],
            )
            save_comment_btn.click(fn=save_comment, inputs=[last_run_id, comment_box], outputs=[comment_saved])

        # ── History tab ───────────────────────────────────────────────────
        with gr.Tab('History') as history_tab:
            gr.Markdown('## Compression Run History')
            with gr.Row():
                refresh_btn = gr.Button('Refresh', variant='secondary')
                delete_btn  = gr.Button('Delete Selected Row', variant='stop')

            with gr.Accordion('Column visibility', open=False):
                col_picker = gr.CheckboxGroup(choices=_ALL_COLS, value=_DEFAULT_COLS, label=None)

            with gr.Row():
                avg_quality = gr.Textbox(label='Avg Quality Score',     interactive=False)
                avg_ratio   = gr.Textbox(label='Avg Compression Ratio', interactive=False)
            history_table = gr.DataFrame(label='Past Runs — click a row to see its diff', interactive=False)
            delete_status = gr.Textbox(label='Status', value='Click a row to select it.', interactive=False)
            gr.Markdown('### Side-by-side Diff')
            diff_panel  = gr.HTML(value='')
            selected_id = gr.State(value=None)

            _outputs = [history_table, avg_quality, avg_ratio, diff_panel]
            refresh_btn.click(fn=load_history, inputs=[col_picker], outputs=_outputs)
            history_tab.select(fn=load_history, inputs=[col_picker], outputs=_outputs)
            col_picker.change(fn=load_history, inputs=[col_picker], outputs=_outputs)
            history_table.select(fn=on_row_select, inputs=[history_table], outputs=[selected_id, diff_panel, delete_status])
            delete_btn.click(
                fn=delete_selected,
                inputs=[selected_id, col_picker],
                outputs=[history_table, avg_quality, avg_ratio, selected_id, diff_panel, delete_status],
            )

    return app

In [ ]:
# Launch UI

app = build_app()
app.launch(share=PUBLIC_UI, server_port=SERVER_PORT, debug = True)

## Step 10 — Programmatic demo (no UI needed)

Run this cell to compress a sample text directly and inspect all metrics inline.

In [ ]:
SAMPLE_TEXT = """
The transformer architecture, introduced in the seminal paper Attention Is All You Need by Vaswani et al.
in 2017, fundamentally changed how we approach sequence modelling tasks in natural language processing.
Prior to transformers, recurrent neural networks (RNNs) and long short-term memory (LSTM) networks were
the dominant architectures for tasks such as machine translation, text summarisation, and question answering.
However, these models suffered from several limitations: they processed tokens sequentially, making
parallelisation difficult; they struggled to capture long-range dependencies due to vanishing gradients;
and training was slow even on modern hardware. The transformer addressed all of these issues through
its self-attention mechanism, which allows every token in a sequence to directly attend to every other
token in a single operation. Multi-head attention further extends this by running several attention
functions in parallel, capturing different types of relationships between tokens simultaneously.
Position encodings are added to token embeddings to give the model a sense of sequence order, since
unlike RNNs the architecture has no inherent notion of position. Feed-forward sub-layers, layer
normalisation, and residual connections complete each transformer block. The result is a model that
trains faster, scales better with data and compute, and generalises more effectively than its
predecessors, setting the stage for large language models like GPT, BERT, and the entire modern
LLM ecosystem.
""".strip()

TARGET = 150  # token budget

input_tok_count = count_tokens(SAMPLE_TEXT)
print(f'Input tokens : {input_tok_count}')
print(f'Target tokens: {TARGET}')
print(f'Status       : {"ready to compress" if input_tok_count > TARGET else "already within budget"}')
print()

t0 = time.perf_counter()
compressed, in_tok, out_tok = compress(SAMPLE_TEXT, TARGET)
elapsed = round((time.perf_counter() - t0) * 1000, 1)

score = semantic_score(SAMPLE_TEXT, compressed)
ratio = round(out_tok / in_tok, 4)

print('─' * 60)
print(compressed)
print('─' * 60)
print(f'Output tokens    : {out_tok}')
print(f'Compression ratio: {ratio}')
print(f'Quality score    : {score}')
print(f'Duration         : {elapsed} ms')
print(f'Model            : {get_current_model_id()}')
print(f'Tokenizer        : {get_current_tokenizer_id()}')